In [1]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.abspath('../Scripts'))
sys.path.insert(0, os.path.expanduser('~/CDD_Vault_API/python'))  # CDD Vault API (get_df)

/home/gtamo/MS_ML


In [2]:
%load_ext autoreload
%autoreload 2

import re
import os
import pandas as pd
import numpy as np
import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import csv
import pickle

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem,rdFMCS
# import prolif as plf
from glob import glob
import meeko
import subprocess as sub
# from vina import Vina
import time
from tqdm.auto import tqdm  # auto -> notebook/VSCode widget
tqdm.pandas()
import importlib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, silhouette_score, davies_bouldin_score, calinski_harabasz_score
from xgboost import XGBClassifier,XGBRegressor
from openTSNE import TSNE as oTSNE        # pip install openTSNE
import seaborn as sns
from scipy import stats
import pyarrow.dataset as pads
import csv, contextlib, threading, joblib
import joblib
from joblib import Parallel, delayed
from datetime import date
import ipywidgets as w
from IPython.display import clear_output, display

# user defined modules
import Rdkit_tools as rdkit_tools
importlib.reload(rdkit_tools)
import Molecule as M
import ML_Reg as ML_Reg
import ML_Class as ML_Class
from MolViz3D import MolViz3D
import Statistics_tools as stats_tools
import python.functions as fn
from get_library import get_df   # CDD Vault collection export
from python.MS_build_ML import PARAMS, DATA, OUTPUT   # CLI classes reused here (%autoreload picks up edits)
# from tdc.multi_pred import DTI

In [9]:
## params — every YAML key becomes an attribute (params.MS_PATH); Dropbox readers + MLTrail vault bound here
CONFIG = 'config/config.yaml'

params = PARAMS(CONFIG)
params.load_params()
params.setup_dropbox()    # params.dbx / params.dbx_glob per DROPBOX_BACKEND
params.load_registry()    # params.registry (shared MLTrail vault)

## transition shim — keeps the bare YAML names (SERAC_C, MS_PATH, ...) working in the cells below
globals().update({k: v for k, v in params.__dict__.items() if k.isupper()})
registry = params.registry
_dbx = _fbx_open = params.dbx
_fbx_glob = params.dbx_glob

print('> chemlib overwrite:', params.CHEMLIB_OVERWRITE, '| dfraw overwrite:', params.DFRAW_OVERWRITE)
print('> features:', params.FEATURES_TYPE)

> loaded 82 params from config/config.yaml
> Dropbox backend: rclone
> chemlib overwrite: False | dfraw overwrite: False
> features: H237


## 0. Imports

In [14]:
%%time
## data — library + proteomics + chemistry features (see python/MS_build_ML.py:DATA)
data = DATA()
data.load_chemical_lib_df(params)                                  # serac_df + validated / devalidated targets
## cm2rm now comes from config CM2RM_PARTS — any subset of contaminants (the CONTAMINANTS csv) |
## control_compounds (CONTROLS) | fbx_independent (Px_Ligase_dependent == 0 or Px_validated_WT == 0)
data.get_contaminants_and_controls(params)                         # parts=(...) here would override the config
data.load_proteomics_data(params)                                  # DFRAW_OVERWRITE -> rebuild from the tranches, else read the parquets
data.compute_features(params)                                      # FEATURES_TYPE -> MF_features (+ MF morgan bits)

## OpenTargets disease scores + the big-pharma-relevant subset (see python/MS_build_ML.py:DATA)
data.load_opentargets(params)   # ot_df, cached to OT_CACHE — delete that file to force a recompute from OT_ROOT
data.filter_pharma_targets()    # ot_pharma, via the module-level PHARMA_PRIORITY_AREAS / PHARMA_DROP_AREAS

## transition shim — bare names used by the cells below
serac_df, MS, df_raw, MF, MF_features = data.serac_df, data.MS, data.df_raw, data.MF, data.MF_features
control_compounds, contaminants, cm2rm = data.control_compounds, data.contaminants, data.cm2rm
fbx_independent = data.fbx_independent
validated_targets, devalidated_targets = data.validated_targets, data.devalidated_targets
ot_df, ot_pharma = data.ot_df, data.ot_pharma
# ot_df.to_csv('data/patent/20260609_OpenTarget_list.csv', sep=',', index=False)

> Chemical lib dim: (14168, 13) | 40 validated / 2772 devalidated targets
> 2 control + 81 contaminant compounds (from config) + 1026 fbx independent | cm2rm = contaminants + fbx_independent = 1107
> loaded cached MS (7222, 5) | df_raw (66186971, 12)
> DROP_CM2RM: dropped 1,034 compounds (10,468,924 df_raw rows) / 996 MS compounds
> MS derived from df_raw: 6,226 compounds | 99 change activity bin | 2 keep the delivered value (no df_raw row)
> checking smiles


100%|██████████| 14168/14168 [00:03<00:00, 4096.29it/s]


> FEATURES_TYPE: H237


H237 descriptors: 100%|██████████| 128/128 [00:08<00:00, 15.67it/s]


> 0 smiles failed | MF (14168, 2049) | MF_features (14086, 4470)
> loaded cached output/MS/opentargets_target_disease.parquet
> 232,004 (target, disease) rows / 11,584 targets resolved
> 78,935 / 232,004 (target, disease) rows pass the pharma filter | 9,105 unique targets
  17,071  cancer or benign tumor
   3,890  cardiovascular disease
   2,504  disorder of visual system
   3,466  endocrine system disease
   3,416  hematologic disease
   3,938  immune system disease
   3,710  infectious disease
   8,708  musculoskeletal or connective tissue disease
  25,393  nervous system disease
   9,849  nutritional or metabolic disease
   1,704  psychiatric disorder
   2,357  respiratory or thoracic disease
CPU times: user 1min 18s, sys: 36.1 s, total: 1min 55s
Wall time: 1min 38s


## 0. Checking Hot targets in Primary screen 

In [15]:
len(data.cm2rm)

1107

In [17]:
## checking single and low volcanoes
ACT_BINS = ['Single (1)', 'Low (2-10)']   # MS activity classes kept

# hot target list: genes = the df_raw symbol
hot_df = pd.read_csv(params.HOT_TARGETS)

# single / low compounds from MS, without the devalidated ones (cm2rm)
cmps = data.MS.loc[data.MS['activity'].isin(ACT_BINS) & ~data.MS['compound'].isin(data.cm2rm), ['compound', 'activity']]

# significant down hits on a hot target, for those compounds only
MS_hot_df = (data.df_raw[data.df_raw['genes'].isin(hot_df['genes'])]
             .query('significant == 1 and logfc < 0')
             .merge(cmps, on='compound')
             .rename(columns={'MSPlate': 'plate'})
             [['compound', 'genes', 'activity', 'logfc', 'pvalue', 'plate']]
             .sort_values(['genes', 'logfc']).reset_index(drop=True))
print(MS_hot_df.shape)
MS_hot_df.sort_values('pvalue').head(3)

(126, 6)


,compound,genes,activity,logfc,pvalue,plate
19,SRB-0005194,GSPT1,Low (2-10),-1.373734,2.519032e-46,Pw151
23,SRB-0000870,GSPT1,Low (2-10),-0.852902,5.661855e-43,EM_S00017_Q1_L
18,SRB-0003613,GSPT1,Low (2-10),-1.387920,9.422559e-42,Pw154


In [18]:
## volcano of each MS_hot_df row: its experiment (compound x plate), the hot gene ringed
OUT_DIR = 'output/MS/volcanoes/hot_single_low'
XLIM    = (-8.0, 8.0)    # logfc axis of the Px interface volcanoes
DPI     = 150

# experiment of each row: the df_raw row with the same compound, gene, plate, logfc and pvalue
keys = ['compound', 'genes', 'plate', 'logfc', 'pvalue']
raw = (data.df_raw.loc[data.df_raw['compound'].isin(MS_hot_df['compound']),
                       ['compound', 'genes', 'MSPlate', 'logfc', 'pvalue', 'significant', 'uniquecontrast', 'MoleculeBatchID']]
       .rename(columns={'MSPlate': 'plate'}))
hits = MS_hot_df.merge(raw[keys + ['uniquecontrast', 'MoleculeBatchID']], on=keys, how='left')

# volcano source: every gene of those experiments, p = 0 floored to the smallest non-zero p (as the Px interface)
pmin = data.df_raw.loc[data.df_raw['pvalue'] > 0, 'pvalue'].min()
vol = raw[raw['uniquecontrast'].isin(hits['uniquecontrast'])].assign(pvalue=lambda d: d['pvalue'].clip(lower=pmin))

# one png per row, titled as in the Px interface
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
for r in tqdm(hits.itertuples(), total=len(hits)):
    fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=DPI)
    agg = fn.plot_volcano_significant(vol, r.uniquecontrast, r.genes, xmin=XLIM[0], xmax=XLIM[1], ax=ax, title='')
    ax.get_legend().remove()
    ax.set_ylim(bottom=0)
    ax.set_title(f'{r.genes} · {r.MoleculeBatchID} · {r.plate} · {r.activity} (logfc {r.logfc:.2f}) ({len(agg):,} genes)',
                 loc='left', fontsize=8, fontweight='bold')
    fig.savefig(f'{OUT_DIR}/{r.genes}_{r.uniquecontrast}.png', bbox_inches='tight')
    plt.close(fig)

hits.head(3)


  0%|          | 0/126 [00:00<?, ?it/s]

,compound,genes,activity,logfc,pvalue,plate,uniquecontrast,MoleculeBatchID
0,SRB-0008516,BCL6,Low (2-10),-0.348054,0.000030,EM_S00022_Q1_L,SRB_0008516_EM_S00022_Q1_L,SRB-0008516-001
1,SRB-0001914,BRAF,Low (2-10),-2.373101,0.000002,Plate4,X1914.001_vs_DMSO_Plate4,SRB-0001914-001
2,SRB-0005608,BRAF,Low (2-10),-1.622024,0.000056,Pw71,SRB.0005608.001_vs_SRB.0005608.001_complement_...,SRB-0005608-001


## 1. Single Target view - Finding under-rated compounds to enrich around

In [ ]:
## get hot target list: genes_ori = the name on the list, genes = the df_raw symbol
hot_df = pd.read_csv(params.HOT_TARGETS)
print(f'> hot targets: {hot_df.shape[0]:,} rows x {hot_df.shape[1]} cols')

## get SAR predictability per gene
gene_metrics = pd.read_csv(GENE_SAR_OUT)

## match the hot target symbols onto the genes our screen measured; target = the name on the list
hot_map = fn.match_gene_symbols(hot_df['genes'], gene_metrics['gene']).assign(target=hot_df['genes_ori'])
hot_r2  = (hot_map[hot_map['matched']].merge(gene_metrics, left_on='symbol', right_on='gene')
                                      .sort_values('R2', ascending=False))
print(f"> matched {hot_map['matched'].sum()}/{len(hot_map)} targets")
print(f"> never measured: {sorted(hot_map.loc[~hot_map['matched'], 'target'])}")

## + compounds per activity class that significantly down-regulate it, and the canonical MS score
## (max of -logfc * -log10(p) over significant rows — Px_interface.py:127-131 / :386)
hot_tbl = (hot_r2.merge(fn.count_hits_by_activity(data.df_raw, data.MS, genes=hot_r2['symbol']),
                        left_on='symbol', right_on='genes').drop(columns='genes')
                 .merge(fn.gene_ms_score(data.df_raw, genes=hot_r2['symbol']),
                        left_on='symbol', right_index=True, how='left'))
hot_tbl[['target', 'symbol', 'R2', 'n', 'spearman_r', 'ms_score', 'Single (1)', 'Low (2-10)',
         'Medium (11-25)', 'High (>25)', 'hits']].round(2).head(20)

In [ ]:
## Looking for gene specific under-appreciated compounds — set GENE to any measured target.
## Pareto gate (layer) + isolation (d_any), over the volcanoes where GENE is NOT significant.
GENE     = 'SMARCA2'   # any symbol in data.df_raw['genes'], e.g. PCSK9 / APOL1 / KEAP1 / FOXA1
PARTNERS = []        # co-regulated genes to control for (PCSK9: the sterol block); [] skips it

_g = data.df_raw[data.df_raw['genes'] == GENE]
if _g.empty:
    raise ValueError(f'{GENE} is not measured — no rows in df_raw')
print(f'> {GENE}: {_g["uniquecontrast"].nunique():,} experiments | '
      f'{int(((_g["significant"] == 1) & (_g["logfc"] < 0)).sum()):,} significant-down')

standout = fn.rank_gene_standout(data.df_raw, GENE, max_layer=params.STANDOUT_MAX_LAYER,
                                 radius=params.STANDOUT_RADIUS, scale_pct=params.STANDOUT_SCALE_PCT)
_meta = data.df_raw.drop_duplicates('uniquecontrast').set_index('uniquecontrast')
standout['compound'] = standout['uniquecontrast'].map(_meta['compound'])
standout['plate'] = standout['uniquecontrast'].map(_meta['MSPlate'])   # NaN for the 20260429 CDD export
standout = (standout[~standout['compound'].isin(set(data.cm2rm))]
            .merge(data.MS[['compound', 'ndown', 'activity']], on='compound', how='left'))

## control: a true standout moves GENE alone, so its co-regulated partners must stay flat
standout[['partners_down', 'partners_logfc']] = np.nan
if PARTNERS:
    _p = (data.df_raw[data.df_raw['genes'].isin(PARTNERS)
                      & data.df_raw['uniquecontrast'].isin(standout['uniquecontrast'])]
          .assign(_down=lambda z: (z['significant'] == 1) & (z['logfc'] < 0)))
    standout[['partners_down', 'partners_logfc']] = (
        _p.groupby('uniquecontrast').agg(partners_down=('_down', 'sum'),
                                         partners_logfc=('logfc', 'mean'))
          .reindex(standout['uniquecontrast']).to_numpy())

print(f'> {len(standout)} non-significant volcanoes within Pareto layer {params.STANDOUT_MAX_LAYER}'
      + (f' | {int((standout["partners_down"] == 0).sum())} with no partner moving' if PARTNERS else ''))

standout = standout[standout['activity']!='High (>25)']

standout[['compound', 'plate', 'logfc', 'pvalue', 'y', 'n_dom', 'layer', 'd_any', 'd_bulk',
          'density', 'n_frontier', 'ndown', 'activity', 'partners_down', 'partners_logfc']]

In [ ]:
## plot volcano of selectd compound:
## interactive: fill = significance + direction (grey / SILENT_C down / ACTIVE_C up), dark rings + labels
## = Pareto layers 1..STANDOUT_MAX_LAYER (deeper = fainter), blue ring = GENE, structure top-right.
## gold rings = the STRING first shell of GENE; only the symbol leaves the machine, the match is local
## the buttons and the slider move through the standout ranks — no need to edit a value
_shell = fn.string_first_shell(GENE, species=params.STRING_SPECIES, min_score=params.STRING_MIN_SCORE,
                               limit=params.STRING_MAX_PARTNERS, cache_dir=params.STRING_CACHE)

_out  = w.Output()
_rank = w.IntSlider(value=0, min=0, max=len(standout) - 1, description='rank',
                    continuous_update=False, layout=w.Layout(width='420px'))
_prev, _next = w.Button(description='< prev'), w.Button(description='next >')

def _show_rank(rank):
    r = standout.iloc[rank]
    with _out:
        clear_output(wait=True)
        print(f"> rank {rank}/{len(standout) - 1}: {r['compound']} {r['plate']} | {GENE} logfc "
              f"{r['logfc']:.3f} p {r['pvalue']:.2e} | layer {int(r['layer'])} "
              f"| d_any {r['d_any']:.3f} | ndown {r['ndown']:.0f}")
        fn.plot_volcano_pareto(data.df_raw, r['uniquecontrast'], highlight=GENE,
                               max_layer=params.STANDOUT_MAX_LAYER, up_color=params.ACTIVE_C,
                               down_color=params.SILENT_C, compound=r['compound'],
                               partners=_shell['partner'].tolist(),
                               struct_dir=params.SRB_PNG_DIR,   # None
                               struct_size=0.45).show()

def _step(delta):
    _rank.value = min(max(_rank.value + delta, _rank.min), _rank.max)

_prev.on_click(lambda _: _step(-1))
_next.on_click(lambda _: _step(+1))
_rank.observe(lambda c: _show_rank(c['new']), names='value')
display(w.HBox([_prev, _next, _rank]), _out)
_show_rank(_rank.value)

## 2. Systematic multi-target approach

In [7]:
## every hot target gets its own HTML report — skip a gene with no standout or only High (>25)
## gold rings = that target's STRING first shell; only the symbol leaves the machine
## plus ONE csv over every target: the html table columns, with `target` first
REPORT_DIR = params.DROPBOX_ML + params.STANDOUT_REPORT_DBX
CSV_NAME   = f'{date.today():%Y%m%d}_Echoes_compounds.csv'
CSV_COLS   = ['target', 'compound', 'plate', 'logfc', 'pvalue', 'layer', 'd_any', 'd_bulk',
              'density', 'n_frontier', 'ndown', 'activity', 'partners_down', 'partners_logfc',
              'rank', 'drawn']   # drawn = its volcano is plotted (the top STANDOUT_REPORT_MAX)
TARGETS    = hot_tbl['symbol'].dropna().unique()
Path(params.STANDOUT_REPORT_LOC).mkdir(parents=True, exist_ok=True)
rows, echoes, skipped = [], [], {}

In [ ]:
## Extract individual reports
for _gene in tqdm(TARGETS, desc='targets'):
    _s   = fn.standout_table(data.df_raw, data.MS, _gene, cm2rm=data.cm2rm,
                             max_layer=params.STANDOUT_MAX_LAYER, radius=params.STANDOUT_RADIUS,
                             scale_pct=params.STANDOUT_SCALE_PCT)
    _sel = _s[_s['activity'] != 'High (>25)'].reset_index(drop=True)
    if not len(_sel):
        skipped[_gene] = 'no standout' if not len(_s) else 'only High (>25)'
        continue
    _shell = fn.string_first_shell(_gene, species=params.STRING_SPECIES,
                                   min_score=params.STRING_MIN_SCORE,
                                   limit=params.STRING_MAX_PARTNERS, cache_dir=params.STRING_CACHE)
    _name = f'{date.today():%Y%m%d}_{_gene}_report.html'
    _loc  = os.path.join(params.STANDOUT_REPORT_LOC, _name)
    Path(_loc).write_text(fn.standout_report_html(
        data.df_raw, _gene, _sel, max_layer=params.STANDOUT_MAX_LAYER, up_color=params.ACTIVE_C,
        down_color=params.SILENT_C, partners=_shell['partner'].tolist(),
        struct_dir=params.SRB_PNG_DIR, max_volcanoes=params.STANDOUT_REPORT_MAX,
        note=f'Gold rings = the {len(_shell)} STRING first-shell partners.'), encoding='utf-8')
    fn.push_rclone(_loc, REPORT_DIR + _name)
    echoes.append(_sel.assign(target=_gene, rank=np.arange(1, len(_sel) + 1),
                              drawn=np.arange(len(_sel)) < (params.STANDOUT_REPORT_MAX or len(_sel))))
    rows.append({'gene': _gene, 'candidates': len(_sel), 'top_d_any': _sel['d_any'].iloc[0],
                 'best_layer': int(_sel['layer'].min()), 'MB': os.path.getsize(_loc) / 1e6,
                 'report': _name})

reports = pd.DataFrame(rows).sort_values('top_d_any', ascending=False).reset_index(drop=True)
echoes  = pd.concat(echoes, ignore_index=True)[CSV_COLS]
_csv    = os.path.join(params.STANDOUT_REPORT_LOC, CSV_NAME)
echoes.to_csv(_csv, index=False)
fn.push_rclone(_csv, REPORT_DIR + CSV_NAME)

print(f'> {len(reports)} reports -> {REPORT_DIR} ({reports["MB"].sum():.0f} MB total) '
      f'| {len(skipped)} genes skipped')
print(f'> {len(echoes):,} rows over {echoes["target"].nunique()} targets '
      f'({echoes["compound"].nunique():,} distinct compounds, {int(echoes["drawn"].sum())} plotted) -> {CSV_NAME}')
print('> skipped:', skipped)
reports.round(3)

In [8]:
echoes = pd.read_csv(params.dbx(REPORT_DIR + CSV_NAME))


In [9]:
echoes

,target,compound,plate,logfc,pvalue,layer,d_any,d_bulk,density,n_frontier,ndown,activity,partners_down,partners_logfc,rank,drawn
0,PCSK9,SRB-0001210,Pw34,-0.394420,1.856863e-06,1,1.641482,1.641482,0,4,0.0,Silent,NaN,NaN,1,True
1,PCSK9,SRB-0011116,Pw183,-0.353285,1.256223e-08,2,0.463749,0.509516,0,6,6.0,Low (2-10),NaN,NaN,2,True
2,PCSK9,SRB-0009363,Pw95,-0.673564,2.428619e-08,1,0.391048,0.391048,0,6,6.0,Low (2-10),NaN,NaN,3,True
3,PCSK9,SRB-0011879,Pw222,-0.594849,3.958722e-09,2,0.344598,0.427344,0,7,2.0,Low (2-10),NaN,NaN,4,True
4,PCSK9,SRB-0011416,Pw208,-0.551757,2.862200e-12,2,0.340951,0.799672,0,5,3.0,Low (2-10),NaN,NaN,5,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,SOX4,SRB-0010468,Pw113,-1.407624,1.165221e-01,2,0.412497,0.412497,0,4,1.0,Single (1),NaN,NaN,9,True
174,SOX4,SRB-0008754,Pw113,-1.819252,4.417628e-02,1,0.087706,0.523675,1,11,19.0,Medium (11-25),NaN,NaN,10,True
175,SOX4,SRB-0008225,Pw101,-2.234154,5.477380e-03,1,0.082456,0.082456,1,4,7.0,Low (2-10),NaN,NaN,11,False
176,STING1,SRB-0002263,Plate45_Pw1,-1.425062,9.194816e-03,2,0.112546,0.234519,3,8,0.0,Silent,NaN,NaN,1,True
